# FairFace gender classification
Run the cells in order. Add FairFace as a Kaggle Input or place/mount it under `/content/data` on Colab. No dataset slug or download URL is assumed.

In [ ]:
import os, sys, subprocess, torch
from pathlib import Path
print('torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
try:
    import torchvision; print('torchvision:', torchvision.__version__)
except Exception as e: print('torchvision import:', e)
try: print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout)
except Exception as e: print(e)
try:
    import psutil; print('RAM GiB:', psutil.virtual_memory().total/2**30)
except ImportError: pass
print('Disk:', __import__('shutil').disk_usage('/').free/2**30, 'GiB free')

## Source code
For a cloned project, replace `<repo_url>` below with your repository URL. If the notebook is already running from the project directory, leave `PROJECT` unchanged.

In [ ]:
PROJECT = Path.cwd()
if not (PROJECT/'src').exists():
    REPO_URL = '<repo_url>'
    if REPO_URL.startswith('http'): subprocess.run(['git','clone',REPO_URL,'/content/fairface_gender'],check=True)
    PROJECT = Path('/content/fairface_gender')
os.chdir(PROJECT); sys.path.insert(0,str(PROJECT))
print('Project:', PROJECT)

In [ ]:
from src.env import detect_env, resolve_paths
print('Environment:', detect_env())
paths=resolve_paths({'data_dir':None,'output_dir':None}); print(paths)

## Data and splits
On Colab, mount Drive if your data or outputs are there. Copying a compressed dataset to `/content/data` can improve image loading speed. Kaggle data should be attached through Add Input.

In [ ]:
if detect_env() == 'colab':
    try:
        from google.colab import drive; drive.mount('/content/drive')
    except ImportError: pass
DATA_DIR = None  # optionally set a concrete path
subprocess.run([sys.executable,'scripts/prepare_splits.py'] + (['--data_dir',DATA_DIR] if DATA_DIR else []),check=True)

## Smoke run (optional)
This runs two epochs per model on the available training split. Use one common epoch count for all models in the final experiment.

In [ ]:
for name in ['resnet50_base','resnet50_head','resnet50_se','resnet50_cbam']:
    subprocess.run([sys.executable,'-m','src.train','--model',name,'--config','configs/default.yaml','--seed','42','--resume','--debug'],check=True)

## Full training
Each model runs independently. Re-run interrupted cells with `--resume`; runs marked DONE are skipped.

In [ ]:
subprocess.run([sys.executable,'-m','src.train','--model','resnet50_base','--config','configs/default.yaml','--seed','42','--resume'],check=True)

In [ ]:
subprocess.run([sys.executable,'-m','src.train','--model','resnet50_head','--config','configs/default.yaml','--seed','42','--resume'],check=True)

In [ ]:
subprocess.run([sys.executable,'-m','src.train','--model','resnet50_se','--config','configs/default.yaml','--seed','42','--resume'],check=True)

In [ ]:
subprocess.run([sys.executable,'-m','src.train','--model','resnet50_cbam','--config','configs/default.yaml','--seed','42','--resume'],check=True)

## Evaluation, comparison, and archive

In [ ]:
models=['resnet50_base','resnet50_head','resnet50_se','resnet50_cbam']
for name in models: subprocess.run([sys.executable,'-m','src.evaluate','--model',name,'--seed','42'],check=True)
subprocess.run([sys.executable,'-m','src.compare','--models',*models],check=True)
from IPython.display import display, Image
display(Image(filename='outputs/compare/roc.png'))

In [ ]:
import shutil
shutil.make_archive('fairface_outputs','zip','outputs')
print('Created fairface_outputs.zip')